# Colab: Quantization (simple)

Lives in **`9_QUANTIZATION/`**. Run in **Google Colab** with a **GPU** (T4 is fine).

**Idea:** load a model with **fewer bits per weight** (4-bit) so it fits in memory.

Notes: [`what_is_quantization.md`](./what_is_quantization.md)

**Pattern:** code cell first → explanation under it.

Needs: Colab Secret `HF_TOKEN` (optional for TinyLlama; required for gated Llama).

## Setup

In [ ]:
!pip -q install transformers accelerate bitsandbytes

### Explanation — install

| Package | Why |
|---------|-----|
| `transformers` | Load Hub models + `BitsAndBytesConfig` |
| `accelerate` | Helps place the model on GPU (`device_map`) |
| `bitsandbytes` | Does the **4-bit / 8-bit** loading |

In [ ]:
import torch

try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("HF_TOKEN: logged in")
    else:
        print("HF_TOKEN: not set")
except Exception as e:
    print("Secrets / login skipped:", e)

print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

### Explanation — GPU + token

bitsandbytes 4-bit needs a **CUDA GPU**.  
`HF_TOKEN` matters if you switch to a **gated** model (e.g. Llama).

## 1. Make a 4-bit config

This is the heart of quantization in Hugging Face.


In [ ]:
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(quant_config)

### Explanation — `BitsAndBytesConfig`

Normally weights are big (often 32-bit floats).  
This config says: **load them in 4-bit** instead.

| Setting | Meaning |
|---------|---------|
| `load_in_4bit=True` | Store weights with **4 bits** each |
| `bnb_4bit_use_double_quant=True` | Quantize again (saves a bit more memory) |
| `bnb_4bit_compute_dtype=torch.bfloat16` | Do the **math** in `bfloat16` (fast on GPU) |

```text
weights stored:  4-bit (small in memory)
math runs in:    bfloat16 (good GPU compute)
```

You could use `load_in_8bit=True` instead for 8-bit (don’t mix both styles in one config).

This object alone does nothing yet — next we **pass it** when loading a model.

## 2. Load the model in 4-bit + generate

Uses **TinyLlama** (public, small) so free Colab works easily.  
Same pattern as loading Llama — only the model id changes.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
)

prompt = "What is quantization in one short sentence?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Explanation — load + run

| Line | Meaning |
|------|---------|
| `AutoTokenizer.from_pretrained` | Matching tokenizer (same as tokenizers notebook) |
| `quantization_config=quant_config` | Apply **4-bit** loading from section 1 |
| `device_map="auto"` | Put layers on GPU automatically |
| `tokenizer(...)` | Text → token ids |
| `model.generate(...)` | Model writes the next tokens |
| `decode(...)` | Ids → text |

```text
without quant:  heavy weights in memory
with 4-bit:     same model idea, much lighter load
```

#### Course-style (Llama) — same code, different id

If you have Llama Hub access + `HF_TOKEN`:

```python
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
```

Keep the same `quant_config` — that is the point of quantization: **big models become runnable** on a T4.

#### Link to your notes

Quantization → lighter load.  
Later: **QLoRA** = fine-tune while staying quantized.


## 3. Memory footprint (why quantization matters)

Same loaded model — now measure how much memory it uses.


In [ ]:
# Hugging Face helper (bytes the model weights take)
footprint_bytes = model.get_memory_footprint()
print(f"Model memory footprint: {footprint_bytes / 1e6:.1f} MB")
print(f"Model memory footprint: {footprint_bytes / 1e9:.3f} GB")

# What GPU currently holds (allocated)
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated()
    reserved = torch.cuda.memory_reserved()
    print(f"GPU memory allocated: {allocated / 1e9:.3f} GB")
    print(f"GPU memory reserved:  {reserved / 1e9:.3f} GB")

# Rough “if this were full float32” estimate (for comparison only)
num_params = sum(p.numel() for p in model.parameters())
fp32_estimate = num_params * 4  # 4 bytes per param
print(f"Approx params: {num_params / 1e9:.3f} B")
print(f"Rough FP32 size if not quantized: ~{fp32_estimate / 1e9:.2f} GB")
print("(4-bit footprint should look much smaller than that FP32 estimate)")

### Explanation — memory footprint

#### What we print

| Call | Meaning |
|------|---------|
| `model.get_memory_footprint()` | HF estimate of how big the **loaded weights** are (bytes) |
| `torch.cuda.memory_allocated()` | How much GPU memory PyTorch is **using now** |
| `torch.cuda.memory_reserved()` | GPU memory **reserved** by the caching allocator (often ≥ allocated) |
| `num_params * 4` | Rough size if every param were **float32** (4 bytes each) — comparison only |

#### Why show this?

Quantization’s whole point is **smaller memory**:

```text
FP32 estimate (big)     →  “what full precision might cost”
4-bit footprint (small) →  “what we actually loaded”
```

You should see the **4-bit footprint much lower** than the rough FP32 estimate.

#### Notes

- Numbers vary by model size, GPU, and what else is loaded  
- TinyLlama is small, so savings look modest in GB — on **7B/8B Llama** the difference is dramatic  
- Run this **after** `from_pretrained(...)` (section 2) so `model` exists  

```text
Quantization success check ≈ smaller memory footprint for the same model id
```


## Recap

1. Quantization = fewer bits per weight when loading  
2. `BitsAndBytesConfig(load_in_4bit=True, ...)` sets the rules  
3. Pass it as `quantization_config=...` into `from_pretrained`  
4. Check with `model.get_memory_footprint()` (+ GPU allocated)  
5. Needs GPU + `bitsandbytes`  

More detail: [`what_is_quantization.md`](./what_is_quantization.md)